# Kaggle: Curriculum retraining (company-isolated)

This notebook trains the retriever using curriculum grouped by company to prevent cross-company contamination.

It expects you to upload the training JSON manifest and the raw EDGAR HTML files for the companies (e.g., AAPL and AMZN) into the Kaggle notebook input.

link to Kaggle notebook : https://www.kaggle.com/code/ayushdhoble/notebooka9fb13e960/notebook

In [1]:
# Install dependencies
%pip install -q sentence-transformers transformers accelerate datasets faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Copy and rename uploaded files from /kaggle/input into /kaggle/working
import os, shutil, glob

IN = '/kaggle/input'
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)
found = {}

for p in glob.glob('/kaggle/input/**/*', recursive=True):
    if os.path.isfile(p):
        name = os.path.basename(p)
        
        # Check for our target extensions
        if name.lower().endswith(('.json', '.html', '.htm')):
            
            # Map the messy filenames to the clean ones your code expects
            if 'AAPL' in name: 
                clean_name = 'AAPL_10k.html'
            elif 'AMZN' in name: 
                clean_name = 'AMZN_10k.html'
            elif 'GOOGL' in name: 
                clean_name = 'GOOGL_10k.html'
            elif 'NVDA' in name: 
                clean_name = 'NVDA_10k.html'
            elif 'TSLA' in name: 
                clean_name = 'TSLA_10k.html'
            else: 
                clean_name = name # Keeps the original name for your JSON file
            
            tgt = os.path.join(OUT, clean_name)
            try:
                shutil.copy(p, tgt)
                found[name] = tgt
            except Exception as e:
                print(f"Error copying {name}: {e}")

print('Copied and renamed files:', found)
print('Working dir listing:')
print(sorted(os.listdir(OUT)))

Copied and renamed files: {'AMZN_)primary-document.htm': '/kaggle/working/AMZN_10k.html', 'NVDA_primary-document.htm': '/kaggle/working/NVDA_10k.html', 'TSLA_primary-document.htm': '/kaggle/working/TSLA_10k.html', 'GOOGL_primary-document.htm': '/kaggle/working/GOOGL_10k.html', 'AAPL_primary-document.htm': '/kaggle/working/AAPL_10k.html'}
Working dir listing:
['AAPL_10k.html', 'AMZN_10k.html', 'GOOGL_10k.html', 'NVDA_10k.html', 'TSLA_10k.html', '__notebook__.ipynb']


In [3]:
# Write the curriculum trainer script into working folder (if not present)
script_target = '/kaggle/working/kaggle_train_curriculum_retriever.py'
if not os.path.exists(script_target):
    script_text = r""


In [4]:
# (continued) finish writing script_text and save
script_text += r""


In [5]:
# Write the curriculum trainer script into working folder (if not present)
script_target = '/kaggle/working/kaggle_train_curriculum_retriever.py'
if not os.path.exists(script_target):
    script_text = r'''
"""
Kaggle curriculum training script for FinAnalyst.

This is the version you asked for:
- each sample Q&A is grounded in a specific SEC filing
- the record keeps source_file / company / filing_year metadata
- training runs in curriculum order, so you can train on one company
  profile, save the checkpoint, then continue retraining on the next
  company profile

Expected record schema
----------------------
Minimum retriever training record:
{
    "query": "...",
    "golden_context": "exact SEC snippet supporting the answer",
    "source_file": "AAPL_2024_10k.html",
    "company": "AAPL",
    "filing_year": 2024,
    "section_name": "Item 7. Management's Discussion and Analysis",
    "temporal_hard_negative": "optional distractor snippet",
    "final_answer": "optional"
}

Also supported:
- legacy keys: positive, hard_negative
- instruction format: instruction, input, output

What this script trains
-----------------------
1) Optional domain adaptation on raw SEC text using denoising autoencoder.
2) Supervised retriever fine-tuning using either:
   - MNRL pairs
   - triplets with hard negatives
   - hybrid curriculum: triplet first, then MNRL
3) Curriculum order by company, source_file, or company_year.

Typical Kaggle flow
-------------------
1) Upload the JSON training manifest(s) plus optional raw SEC files.
2) Run this script on Kaggle GPU.
3) Download the exported zip.
4) Unzip locally and point src/config.py BASE_ENCODER_MODEL to the folder.
"""

from __future__ import annotations

import argparse
import json
import random
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Iterable, NamedTuple, Sequence

import numpy as np
import pandas as pd
import torch
from sentence_transformers import InputExample, SentenceTransformer
from sentence_transformers import datasets
from sentence_transformers import losses
from torch.utils.data import DataLoader


# -----------------------------
# Reproducibility
# -----------------------------


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


def device_name() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------------
# I/O helpers
# -----------------------------


def _normalize_text(value: object) -> str:
    if value is None:
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "none", "null"}:
        return ""
    return text


def _read_json(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    if isinstance(payload, dict):
        payload = payload.get("data", [])
    if not isinstance(payload, list):
        raise ValueError(f"Unsupported JSON structure in {path}")
    return [x for x in payload if isinstance(x, dict)]


def _read_csv(path: Path) -> list[dict]:
    return pd.read_csv(path).to_dict(orient="records")


def load_records(paths: Sequence[str]) -> list[dict]:
    records: list[dict] = []
    for raw in paths:
        p = Path(raw)
        if not p.exists():
            filing_year: str = ""
            section_name: str = ""
            document_id: str = ""

@property
def group_company(self) -> str:
    return self.company or "UNKNOWN_COMPANY"

@property
def group_source_file(self) -> str:
    return self.source_file or "UNKNOWN_SOURCE"

@property
def group_company_year(self) -> str:
    base = self.company or "UNKNOWN_COMPANY"
    year = self.filing_year or "UNKNOWN_YEAR"
    return f"{base}_{year}"


def normalize_rows(records: Iterable[dict]) -> list[TrainingRow]:
    rows: list[TrainingRow] = []
    for r in records:
        query = _normalize_text(r.get("query"))
        positive = _normalize_text(
            r.get("golden_context")
            or r.get("positive")
            or r.get("context")
            or r.get("supporting_text")
        )
        hard_negative = _normalize_text(
            r.get("temporal_hard_negative")
            or r.get("hard_negative")
            or r.get("negative")
        )

        if not query and r.get("instruction"):
            instruction = _normalize_text(r.get("instruction"))
            input_text = _normalize_text(r.get("input"))
            output_text = _normalize_text(r.get("output"))
            query = f"{instruction}\n{input_text}".strip()
            positive = output_text or positive

        if not query or not positive:
            continue

        rows.append(
            TrainingRow(
                query=query,
                positive=positive,
                hard_negative=hard_negative,
                source_file=_normalize_text(r.get("source_file") or r.get("source") or r.get("file_path")),
                company=_normalize_text(r.get("company") or r.get("ticker") or r.get("company_ticker")),
                filing_year=_normalize_text(r.get("filing_year") or r.get("year") or r.get("fiscal_year")),
                section_name=_normalize_text(r.get("section_name") or r.get("section") or r.get("item_name")),
                document_id=_normalize_text(r.get("document_id") or r.get("doc_id") or r.get("id")),
            )
        )

    # Exact de-duplication on query/positive only.
    seen: set[tuple[str, str]] = set()
    deduped: list[TrainingRow] = []
    for row in rows:
        key = (row.query, row.positive)
        if key in seen:
            continue
        seen.add(key)
        deduped.append(row)
    return deduped


# -----------------------------
# Optional domain adaptation
# -----------------------------


def build_domain_sentences(raw_paths: Sequence[str], min_words: int = 12) -> list[str]:
    sentences: list[str] = []
    for raw in raw_paths:
        p = Path(raw)
        if not p.exists():
            continue
        if p.is_dir():
            for child in sorted(p.glob("**/*")):
                if child.suffix.lower() in {".txt", ".json", ".csv"}:
                    sentences.extend(build_domain_sentences([str(child)], min_words=min_words))
            continue

        if p.suffix.lower() == ".txt":
            for line in p.read_text(encoding="utf-8", errors="ignore").splitlines():
                text = line.strip()
                if len(text.split()) >= min_words:
                    sentences.append(text)
        elif p.suffix.lower() == ".json":
            for rec in _read_json(p):
                for key in ("text", "chunk_text", "golden_context", "positive", "context"):
                    text = _normalize_text(rec.get(key))
                    if len(text.split()) >= min_words:
                        sentences.append(text)
        elif p.suffix.lower() == ".csv":
            df = pd.read_csv(p)
            for col in ("text", "chunk_text", "golden_context", "positive", "context"):
                if col in df.columns:
                    for text in df[col].dropna().astype(str).tolist():
                        cleaned = text.strip()
                        if len(cleaned.split()) >= min_words:
                            sentences.append(cleaned)

    deduped: list[str] = []
    seen: set[str] = set()
    for s in sentences:
        if s not in seen:
            seen.add(s)
            deduped.append(s)
    return deduped


# -----------------------------
# Curriculum grouping
# -----------------------------


def curriculum_key(row: TrainingRow, group_by: str) -> str:
    if group_by == "company":
        return row.group_company
    if group_by == "source_file":
        return row.group_source_file
    if group_by == "company_year":
        return row.group_company_year
    return "ALL"


def sort_group_names(group_names: list[str]) -> list[str]:
    def sort_key(name: str) -> tuple:
        # Try to place yearly groups in ascending year order when possible.
        digits = "".join(ch for ch in name if ch.isdigit())
        year = int(digits[-4:]) if len(digits) >= 4 else 0
        return (name.split("_")[0], year, name)

    return sorted(group_names, key=sort_key)


def split_rows(rows: list[TrainingRow], valid_ratio: float = 0.1, seed: int = 42) -> tuple[list[TrainingRow], list[TrainingRow]]:
    if not rows:
        return [], []
    rng = random.Random(seed)
    idxs = list(range(len(rows)))
    rng.shuffle(idxs)
    n_valid = max(1, int(len(rows) * valid_ratio))
    valid = [rows[i] for i in idxs[:n_valid]]
    train = [rows[i] for i in idxs[n_valid:]]
    return train, valid


# -----------------------------
# Training stages
# -----------------------------


def train_domain_stage(
    base_model: str,
    sentences: list[str],
    output_dir: Path,
    epochs: int,
    batch_size: int,
    max_seq_length: int,
) -> Path:
    if not sentences:
        raise ValueError("No domain sentences found for domain adaptation stage.")

    print(f"[domain] sentences={len(sentences)}")
    model = SentenceTransformer(base_model, device=device_name())
    model.max_seq_length = max_seq_length

    dataset = datasets.DenoisingAutoEncoderDataset(sentences)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=len(sentences) > batch_size)
    loss = losses.DenoisingAutoEncoderLoss(
        model,
        decoder_name_or_path=base_model,
        tie_encoder_decoder=True,
    )
    warmup_steps = max(1, int(len(loader) * epochs * 0.1))

    output_dir.mkdir(parents=True, exist_ok=True)
    model.fit(
        train_objectives=[(loader, loss)],
        epochs=epochs,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
        use_amp=torch.cuda.is_available(),
        output_path=str(output_dir),
        checkpoint_path=str(output_dir / "checkpoints"),
        checkpoint_save_steps=500,
    )
    return output_dir


def train_retriever_stage(
    base_model: str,
    rows: list[TrainingRow],
    output_dir: Path,
    epochs: int,
    batch_size: int,
    max_seq_length: int,
    learning_rate: float,
    loss_mode: str,
) -> Path:
    if not rows:
        raise ValueError("No retriever rows were found for this stage.")

    train_rows, valid_rows = split_rows(rows)
    if not train_rows:
        train_rows = rows
        valid_rows = []

    print(f"[retriever] train={len(train_rows)} valid={len(valid_rows)}")
    model = SentenceTransformer(base_model, device=device_name())
    model.max_seq_length = max_seq_length

    pair_examples = [InputExample(texts=[r.query, r.positive]) for r in train_rows]
    pair_loader = DataLoader(
        pair_examples,
        batch_size=batch_size,
        shuffle=True,
        drop_last=len(pair_examples) > batch_size,
    )
    pair_loss = losses.MultipleNegativesRankingLoss(model=model)

    triplet_rows = [r for r in train_rows if r.hard_negative]
    triplet_examples = [InputExample(texts=[r.query, r.positive, r.hard_negative]) for r in triplet_rows]
    triplet_loader = DataLoader(
        triplet_examples,
        batch_size=batch_size,
        shuffle=True,
        drop_last=len(triplet_examples) > batch_size,
    )
    triplet_loss = losses.TripletLoss(model=model)

    if loss_mode == "triplet":
        if not triplet_examples:
            raise ValueError("loss_mode='triplet' requires hard_negative examples.")
        train_objectives = [(triplet_loader, triplet_loss)]
        warmup_steps = max(1, int(len(triplet_loader) * epochs * 0.1))
    elif loss_mode == "mnrl":
        train_objectives = [(pair_loader, pair_loss)]
        warmup_steps = max(1, int(len(pair_loader) * epochs * 0.1))
    else:
        train_objectives = []
        if triplet_examples:
            train_objectives.append((triplet_loader, triplet_loss))
        train_objectives.append((pair_loader, pair_loss))
        warmup_steps = max(1, int(sum(len(loader) for loader, _ in train_objectives) * epochs * 0.1))

    output_dir.mkdir(parents=True, exist_ok=True)
    model.fit(
        train_objectives=train_objectives,
        epochs=epochs,
        warmup_steps=warmup_steps,
        optimizer_params={"lr": learning_rate},
        show_progress_bar=True,
        use_amp=torch.cuda.is_available(),
        output_path=str(output_dir),
        checkpoint_path=str(output_dir / "checkpoints"),
        checkpoint_save_steps=500,
    )

    if valid_rows:
        eval_rows = valid_rows[: min(32, len(valid_rows))]
        q_emb = model.encode([r.query for r in eval_rows], normalize_embeddings=True, convert_to_numpy=True)
        p_emb = model.encode([r.positive for r in eval_rows], normalize_embeddings=True, convert_to_numpy=True)
        sims = np.matmul(q_emb, p_emb.T)
        top1 = sims.argmax(axis=1)
        recall1 = float(np.mean(top1 == np.arange(len(top1))))
        print(f"[retriever] self-retrieval recall@1={recall1:.3f}")

    return output_dir


# -----------------------------
# Curriculum orchestration
# -----------------------------


def build_curriculum(rows: list[TrainingRow], group_by: str) -> list[tuple[str, list[TrainingRow]]]:
    buckets: dict[str, list[TrainingRow]] = defaultdict(list)
    for row in rows:
        key = curriculum_key(row, group_by)
        buckets[key].append(row)

    group_names = sort_group_names(list(buckets.keys()))
    curriculum: list[tuple[str, list[TrainingRow]]] = []
    for name in group_names:
        # Keep examples within a stage stable but random enough.
        group_rows = buckets[name]
        curriculum.append((name, group_rows))
    return curriculum


def zip_folder(folder: Path, zip_path: Path) -> Path:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in folder.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=path.relative_to(folder))
    return zip_path


# -----------------------------
# CLI
# -----------------------------


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Train a source-aware SEC retriever on Kaggle GPU.")
    p.add_argument("--base_model", default="BAAI/bge-base-en-v1.5")
    p.add_argument("--train_paths", nargs="+", required=True, help="JSON/CSV files or directories with QA records.")
    p.add_argument("--domain_paths", nargs="*", default=[], help="Optional raw SEC text files/directories.")
    p.add_argument("--output_dir", default="/kaggle/working/finanalyst_encoder")
    p.add_argument("--zip_name", default="/kaggle/working/finanalyst_encoder.zip")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--domain_epochs", type=int, default=0)
    p.add_argument("--domain_batch_size", type=int, default=8)
    p.add_argument("--epochs", type=int, default=1)
    p.add_argument("--batch_size", type=int, default=16)
    p.add_argument("--max_seq_length", type=int, default=512)
    p.add_argument("--learning_rate", type=float, default=2e-5)
    p.add_argument("--loss_mode", choices=["mnrl", "triplet", "hybrid"], default="hybrid")
    p.add_argument(
        "--curriculum_by",
        choices=["company", "source_file", "company_year", "none"],
        default="company",
        help="How to retrain sequentially across company profiles.",
    )
    p.add_argument(
        "--final_mix_epochs",
        type=int,
        default=1,
        help="Optional final mixed fine-tuning pass on all rows after curriculum stages.",
    )
    return p.parse_args()


def main() -> None:
    args = parse_args()
    set_seed(args.seed)

    train_records = load_records(args.train_paths)
    rows = normalize_rows(train_records)
    print(f"Loaded records: {len(train_records)}")
    print(f"Supervised rows: {len(rows)}")

    if rows:
        # Small sanity print to make Kaggle debugging easier.
        sample = rows[0]
        print("Sample row:")
        print({
            "query": sample.query[:120],
            "source_file": sample.source_file,
            "company": sample.company,
            "filing_year": sample.filing_year,
            "section_name": sample.section_name,
        })

    output_dir = Path(args.output_dir)
    output_dir.parent.mkdir(parents=True, exist_ok=True)

    current_model = args.base_model

    # Stage 1: optional domain adaptation
    if args.domain_epochs > 0:
        domain_sentences = build_domain_sentences(args.domain_paths or args.train_paths)
        domain_dir = output_dir.parent / f"{output_dir.name}_domain"
        current_model = str(
            train_domain_stage(
                base_model=current_model,
                sentences=domain_sentences,
                output_dir=domain_dir,
                epochs=args.domain_epochs,
                batch_size=args.domain_batch_size,
                max_seq_length=args.max_seq_length,
            )
        )

    # Stage 2: curriculum retraining by company / file / year.
    if args.curriculum_by != "none":
        curriculum = build_curriculum(rows, args.curriculum_by)
        print(f"Curriculum stages: {len(curriculum)} grouped by {args.curriculum_by}")
        for idx, (group_name, group_rows) in enumerate(curriculum, start=1):
            stage_dir = output_dir / f"stage_{idx:03d}_{group_name.replace('/', '_').replace(' ', '_')[:80]}"
            print(f"\n=== Stage {idx}/{len(curriculum)}: {group_name} ({len(group_rows)} rows) ===")
            current_model = str(
                train_retriever_stage(
                    base_model=current_model,
                    rows=group_rows,
                    output_dir=stage_dir,
                    epochs=args.epochs,
                    batch_size=args.batch_size,
                    max_seq_length=args.max_seq_length,
                    learning_rate=args.learning_rate,
                    loss_mode=args.loss_mode,
                )
            )

    # Optional final mixed pass to generalize across companies.
    if args.final_mix_epochs > 0:
        print(f"\n=== Final mixed pass on all {len(rows)} rows ===")
        current_model = str(
            train_retriever_stage(
                base_model=current_model,
                rows=rows,
                output_dir=output_dir,
                epochs=args.final_mix_epochs,
                batch_size=args.batch_size,
                max_seq_length=args.max_seq_length,
                learning_rate=args.learning_rate,
                loss_mode=args.loss_mode,
            )
        )
    else:
        # If no final mix requested, keep the last curriculum stage as output.
        output_dir = Path(current_model)

    zip_path = zip_folder(Path(current_model), Path(args.zip_name))
    print("\nTraining complete.")
    print(f"Final model folder: {current_model}")
    print(f"Zip archive: {zip_path}")
    print("\nLocal project step:")
    print("1) Unzip the archive into a local models folder.")
    print("2) Set src/config.py BASE_ENCODER_MODEL to that folder path.")


if __name__ == "__main__":
    main()
'''
    with open(script_target, 'w', encoding='utf-8') as fh:
        fh.write(script_text)
    print('Wrote', script_target)
else:
    print('Script already present:', script_target)

Wrote /kaggle/working/kaggle_train_curriculum_retriever.py


In [6]:
# Quick sanity checks
import json
print('working files:', sorted(os.listdir('/kaggle/working')))
try:
    with open('/kaggle/working/sec_rag_curriculum_seed.json','r',encoding='utf-8') as f:
        data = json.load(f)
    print('Loaded manifest rows:', len(data))
    print('Sample company distribution:', {r.get('company') for r in data[:10]})
except Exception as e:
    print('Could not read manifest:', e)


working files: ['AAPL_10k.html', 'AMZN_10k.html', 'GOOGL_10k.html', 'NVDA_10k.html', 'TSLA_10k.html', '__notebook__.ipynb', 'kaggle_train_curriculum_retriever.py']
Could not read manifest: [Errno 2] No such file or directory: '/kaggle/working/sec_rag_curriculum_seed.json'


In [7]:
# Build a larger company-scoped SEC training dataset
# Output: /kaggle/working/sec_rag_large_dataset.json
import json, math, os, random, re
from collections import Counter, defaultdict
from pathlib import Path
from bs4 import BeautifulSoup

random.seed(42)
OUT_PATH = Path('/kaggle/working/sec_rag_large_dataset.json')
TARGET_PER_COMPANY = 220  # 5 companies => 1100+ rows

COMPANY_ALIASES = {
    'AAPL': ['AAPL', 'APPLE'],
    'AMZN': ['AMZN', 'AMAZON'],
    'TSLA': ['TSLA', 'TESLA'],
    'GOOGL': ['GOOGL', 'GOOG', 'GOOGLE', 'ALPHABET'],
    'NVDA': ['NVDA', 'NVIDIA'],
}
STOPWORDS = {
    'the','and','for','with','that','this','from','were','are','was','have','has','had',
    'its','into','their','they','them','there','which','will','shall','may','can','could',
    'would','should','about','after','before','during','under','over','between','within',
    'company','fiscal','year','years','item','table','risk','page','pages','report'
}
NUM_RE = re.compile(r'(?<![A-Za-z])\$?\(?-?\d[\d,]*(?:\.\d+)?%?\)?')


def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text).strip())


def detect_company(path: Path) -> str | None:
    name = path.name.upper()
    for company, aliases in COMPANY_ALIASES.items():
        if any(alias in name for alias in aliases):
            return company
    return None


def detect_year(path: Path) -> str:
    m = re.search(r'(19|20)\d{2}', path.name)
    return m.group(0) if m else ''


def html_to_text(path: Path) -> str:
    raw = path.read_text(encoding='utf-8', errors='ignore')
    soup = BeautifulSoup(raw, 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    return normalize_text(soup.get_text(' ', strip=True))


def split_sentences(text: str) -> list[str]:
    chunks = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9(])', text)
    cleaned = []
    for chunk in chunks:
        s = normalize_text(chunk)
        if 18 <= len(s.split()) <= 80:
            cleaned.append(s)
    return cleaned


def topic_from_sentence(sentence: str, max_words: int = 8) -> str:
    words = [w.strip(".,;:()[]{}\"'`").lower() for w in sentence.split()]
    words = [w for w in words if w and w not in STOPWORDS and not w.isdigit()]
    if not words:
        words = [w.lower() for w in sentence.split()[:max_words]]
    return ' '.join(words[:max_words])


def numbers_from_sentence(sentence: str) -> list[float]:
    nums = []
    for token in NUM_RE.findall(sentence):
        cleaned = token.replace('$', '').replace(',', '').replace('(', '').replace(')', '').replace('%', '')
        try:
            nums.append(float(cleaned))
        except Exception:
            pass
    return nums


def add_record(rows, company, source_file, filing_year, section_name, query, context, answer, calc_type, trace):
    rows.append({
        'query': query,
        'golden_context': context,
        'temporal_hard_negative': '',
        'calculation_type': calc_type,
        'reasoning_trace': trace,
        'final_answer': answer,
        'source_file': source_file,
        'company': company,
        'filing_year': filing_year,
        'section_name': section_name,
        'document_id': source_file,
    })


html_files = [Path('/kaggle/working') / name for name in os.listdir('/kaggle/working') if name.lower().endswith('.html')]
company_to_files = defaultdict(list)
for p in html_files:
    c = detect_company(p)
    if c:
        company_to_files[c].append(p)

print('Detected company files:')
for c in sorted(company_to_files):
    print(c, [p.name for p in company_to_files[c]])

all_rows = []
company_counts = Counter()

FACT_TEMPLATES = [
    lambda c, t: f"According to {c}'s filing, what does the passage say about {t}?",
    lambda c, t: f"Which SEC passage for {c} supports the statement about {t}?",
    lambda c, t: f"What is the key disclosure in this {c} filing passage about {t}?",
    lambda c, t: f"Summarize the passage from {c}'s filing about {t}.",
]

for company in sorted(company_to_files):
    company_rows = []
    for html_path in sorted(company_to_files[company]):
        filing_year = detect_year(html_path)
        text = html_to_text(html_path)
        sentences = split_sentences(text)
        random.shuffle(sentences)
        seen_queries = set()

        for sentence in sentences:
            topic = topic_from_sentence(sentence)
            source_file = html_path.name
            section_name = 'Unknown Section'
            trace = f'Strict company filter: company={company}; source_file={source_file}; use only this filing.'

            for template in FACT_TEMPLATES:
                if len(company_rows) >= TARGET_PER_COMPANY:
                    break
                query = template(company, topic)
                key = (query, sentence)
                if key in seen_queries:
                    continue
                seen_queries.add(key)
                add_record(
                    company_rows,
                    company,
                    source_file,
                    filing_year,
                    section_name,
                    query,
                    sentence,
                    sentence,
                    'factual_summary',
                    trace,
                )

            if len(company_rows) >= TARGET_PER_COMPANY:
                break

            nums = numbers_from_sentence(sentence)
            if len(nums) >= 2:
                a, b = nums[0], nums[1]
                if a != 0:
                    pct = ((b - a) / abs(a)) * 100.0
                    query = f"In {company}'s filing, if the metric changed from {a:g} to {b:g}, what is the percentage change?"
                    answer = f"{pct:.2f}%"
                    add_record(
                        company_rows,
                        company,
                        source_file,
                        filing_year,
                        section_name,
                        query,
                        sentence,
                        answer,
                        'percent_change',
                        trace,
                    )
            elif len(nums) == 1 and len(company_rows) < TARGET_PER_COMPANY:
                n = nums[0]
                query = f"What numeric value is reported in this {company} SEC passage?"
                add_record(
                    company_rows,
                    company,
                    source_file,
                    filing_year,
                    section_name,
                    query,
                    sentence,
                    f"{n:g}",
                    'number_extraction',
                    trace,
                )

            if len(company_rows) >= TARGET_PER_COMPANY:
                break

    if len(company_rows) < TARGET_PER_COMPANY:
        print(f'Warning: only {len(company_rows)} rows generated for {company}')
    company_counts[company] = len(company_rows)
    all_rows.extend(company_rows[:TARGET_PER_COMPANY])

# De-duplicate on company + query + answer.
unique_rows = []
seen = set()
for row in all_rows:
    key = (row['company'], row['query'], row['final_answer'])
    if key in seen:
        continue
    seen.add(key)
    unique_rows.append(row)

OUT_PATH.write_text(json.dumps(unique_rows, indent=2), encoding='utf-8')
print('Wrote dataset:', OUT_PATH)
print('Total rows:', len(unique_rows))
print('Company counts:', Counter(r['company'] for r in unique_rows))
print('Sample row:', unique_rows[0] if unique_rows else 'none')

Detected company files:
AAPL ['AAPL_10k.html']
AMZN ['AMZN_10k.html']
GOOGL ['GOOGL_10k.html']
NVDA ['NVDA_10k.html']
TSLA ['TSLA_10k.html']
Wrote dataset: /kaggle/working/sec_rag_large_dataset.json
Total rows: 1091
Company counts: Counter({'AAPL': 220, 'GOOGL': 219, 'NVDA': 218, 'AMZN': 217, 'TSLA': 217})
Sample row: {'query': "According to AAPL's filing, what does the passage say about while maintains insurance coverage certain types of claims?", 'golden_context': 'While the Company maintains insurance coverage for certain types of claims, such insurance coverage may be insufficient to cover all losses or all types of claims that may arise.', 'temporal_hard_negative': '', 'calculation_type': 'factual_summary', 'reasoning_trace': 'Strict company filter: company=AAPL; source_file=AAPL_10k.html; use only this filing.', 'final_answer': 'While the Company maintains insurance coverage for certain types of claims, such insurance coverage may be insufficient to cover all losses or all types 

In [8]:
script_path = '/kaggle/working/kaggle_train_curriculum_retriever.py'

with open(script_path, 'r') as f:
    content = f.read()

# Target the exact failing line and add a fallback to the "company" key
old_line = "return row.group_company"
new_line = "return getattr(row, 'group_company', getattr(row, 'company', 'Unknown'))"

if old_line in content:
    content = content.replace(old_line, new_line)
    with open(script_path, 'w') as f:
        f.write(content)
    print("Successfully patched the curriculum_key function!")
else:
    print("Could not find the line. It might already be patched.")

Successfully patched the curriculum_key function!


In [9]:
# Additional hardening patches for the generated curriculum trainer
import re
from pathlib import Path

script_path = Path('/kaggle/working/kaggle_train_curriculum_retriever.py')
content = script_path.read_text(encoding='utf-8')

# 1) Make curriculum_key robust even if the row object only has `company`.
content = content.replace(
    'def curriculum_key(row: TrainingRow, group_by: str) -> str:\n    if group_by == "company":\n        return row.group_company',
    'def curriculum_key(row: TrainingRow, group_by: str) -> str:\n    company = getattr(row, "group_company", getattr(row, "company", "UNKNOWN_COMPANY"))\n    source_file = getattr(row, "group_source_file", getattr(row, "source_file", "UNKNOWN_SOURCE"))\n    filing_year = getattr(row, "group_company_year", f"{company}_{getattr(row, \"filing_year\", \"UNKNOWN_YEAR\")}")\n    if group_by == "company":\n        return company'
)
content = content.replace(
    '    if group_by == "source_file":\n        return row.group_source_file\n    if group_by == "company_year":\n        return row.group_company_year\n    return "ALL"',
    '    if group_by == "source_file":\n        return source_file\n    if group_by == "company_year":\n        return filing_year\n    return "ALL"'
)

# 2) Force a fallback `group_company` attribute for simple row objects.
if 'group_company' not in content:
    content = content.replace(
        'class TrainingRow:',
        'class TrainingRow:\n    pass\n\n# Compatibility shim for row objects that only expose `company`\n'
    )

# 3) Make load_records handle a single JSON file directly and tolerate mixed inputs.
content = re.sub(
    r'def load_records\(paths: Sequence\[str\]\) -> list\[dict\]:.*?return records\n',
    '''def load_records(paths: Sequence[str]) -> list[dict]:
    records: list[dict] = []
    for raw in paths:
        p = Path(raw)
        if not p.exists():
            continue
        if p.is_dir():
            for child in sorted(p.glob("**/*")):
                if child.suffix.lower() in {".json", ".csv"}:
                    records.extend(load_records([str(child)]))
            continue
        if p.suffix.lower() == ".json":
            records.extend(_read_json(p))
        elif p.suffix.lower() == ".csv":
            records.extend(_read_csv(p))
    return records
''',
    content,
    flags=re.S,
)

# 4) Keep the user-provided direct JSON loading behavior if they prefer a single manifest.
content = content.replace(
    '    train_records = load_records(args.train_paths)',
    '    # Prefer a direct JSON load for the Kaggle manifest when available, otherwise fall back to the helper.\n    if len(args.train_paths) == 1 and str(args.train_paths[0]).lower().endswith(".json"):\n        with open(args.train_paths[0], "r", encoding="utf-8") as f:\n            train_records = json.load(f)\n        if isinstance(train_records, dict):\n            train_records = train_records.get("data", [])\n        if not isinstance(train_records, list):\n            raise ValueError("Training manifest must be a JSON array or an object with a data array.")\n    else:\n        train_records = load_records(args.train_paths)'
)

script_path.write_text(content, encoding='utf-8')
print('Applied additional hardening patches to', script_path)

Applied additional hardening patches to /kaggle/working/kaggle_train_curriculum_retriever.py


In [10]:
script_path = '/kaggle/working/kaggle_train_curriculum_retriever.py'

with open(script_path, 'r') as f:
    lines = f.readlines()

# Find the malformed syntax and turn it into a valid string concatenation
for i, line in enumerate(lines):
    if "))_year" in line:
        lines[i] = line.replace("))_year", ")) + '_year'")

with open(script_path, 'w') as f:
    f.writelines(lines)

print("Fixed the dangling _year syntax error!")

Fixed the dangling _year syntax error!


In [11]:
script_path = '/kaggle/working/kaggle_train_curriculum_retriever.py'

with open(script_path, 'r') as f:
    content = f.read()

training_row_def = """
class TrainingRow:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)
        # Automatically handle the company grouping key
        if 'company' in kwargs and 'group_company' not in kwargs:
            self.group_company = kwargs['company']
"""

if "class TrainingRow:" not in content:
    # Prepend it directly to the top of the script
    content = training_row_def + "\n" + content
    
    with open(script_path, 'w') as f:
        f.write(content)
    print("Successfully injected TrainingRow at the top of the script!")
else:
    print("TrainingRow is already present.")

Successfully injected TrainingRow at the top of the script!


In [12]:
script_path = '/kaggle/working/kaggle_train_curriculum_retriever.py'

with open(script_path, 'r') as f:
    lines = f.readlines()

future_imports = []
clean_lines = []

# Separate the __future__ imports from the rest of the code
for line in lines:
    if line.strip().startswith('from __future__ import'):
        future_imports.append(line)
    else:
        clean_lines.append(line)

# Put __future__ imports at the very top, followed by everything else
with open(script_path, 'w') as f:
    f.writelines(future_imports + clean_lines)

print("Moved __future__ imports back to the top!")

Moved __future__ imports back to the top!


In [13]:
script_path = '/kaggle/working/kaggle_train_curriculum_retriever.py'

with open(script_path, 'r') as f:
    content = f.read()

# --- FIX 1: Guard triplet_loader initialization ---
old_loader_code = """    triplet_loader = DataLoader(
        triplet_examples,
        batch_size=batch_size,
        shuffle=True,
        drop_last=len(triplet_examples) > batch_size,
    )"""

new_loader_code = """    if triplet_examples:
        triplet_loader = DataLoader(
            triplet_examples,
            batch_size=batch_size,
            shuffle=True,
            drop_last=len(triplet_examples) > batch_size,
        )
    else:
        triplet_loader = pair_loader"""

if old_loader_code in content:
    content = content.replace(old_loader_code, new_loader_code)
    print("Successfully guarded triplet_loader initialization!")
else:
    print("Warning: Could not find old_loader_code string match.")

# --- FIX 2: Intercept model.fit to filter objectives if empty ---
old_fit_code = "    model.fit("
new_fit_code = """    # Clean up objectives dynamically if no hard negatives are available
    if not triplet_examples:
        train_objectives = [(l, loss) for l, loss in train_objectives if "TripletLoss" not in type(loss).__name__]
        warmup_steps = max(1, int(len(pair_loader) * epochs * 0.1))
    
    model.fit("""

if old_fit_code in content:
    content = content.replace(old_fit_code, new_fit_code)
    print("Successfully guarded model.fit objectives!")
else:
    print("Warning: Could not find model.fit string match.")

# Save out the changes
with open(script_path, 'w') as f:
    f.write(content)

Successfully guarded triplet_loader initialization!
Successfully guarded model.fit objectives!


In [14]:
# Run curriculum trainer (company-isolated)
# Stronger settings for the larger dataset: more epochs and larger batch size for better retrieval quality.
!python /kaggle/working/kaggle_train_curriculum_retriever.py \
  --train_paths /kaggle/working/sec_rag_large_dataset.json \
  --domain_paths /kaggle/working/AAPL_10k.html /kaggle/working/AMZN_10k.html /kaggle/working/TSLA_10k.html /kaggle/working/GOOGL_10k.html /kaggle/working/NVDA_10k.html \
  --output_dir /kaggle/working/finanalyst_encoder \
  --zip_name /kaggle/working/finanalyst_encoder.zip \
  --epochs 4 --batch_size 32 --domain_epochs 0 --loss_mode hybrid --curriculum_by company --final_mix_epochs 3

Loaded records: 1091
Supervised rows: 1091
Sample row:
{'query': "According to AAPL's filing, what does the passage say about while maintains insurance coverage certain types of claims?", 'source_file': 'AAPL_10k.html', 'company': 'AAPL', 'filing_year': '', 'section_name': 'Unknown Section'}
Curriculum stages: 5 grouped by company

=== Stage 1/5: AAPL (220 rows) ===
[retriever] train=198 valid=22
config_sentence_transformers.json: 100%|████████| 124/124 [00:00<00:00, 527kB/s]
README.md: 94.6kB [00:00, 122MB/s]
sentence_bert_config.json: 100%|██████████████| 52.0/52.0 [00:00<00:00, 248kB/s]
config.json: 100%|█████████████████████████████| 777/777 [00:00<00:00, 5.10MB/s]
model.safetensors: 100%|██████████████████████| 438M/438M [00:02<00:00, 163MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 1561.36it/s, Materializing param=
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids 

In [15]:
# Provide download link for resulting zip
from IPython.display import FileLink
import os
zip_path = '/kaggle/working/finanalyst_encoder.zip'
if os.path.exists(zip_path):
    display(FileLink(zip_path))
else:
    print('Zip not found yet:', zip_path)


/kaggle/working/finanalyst_encoder.zip